In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install dagshub
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: dacite
    Found existing installation: dacite 1.9.2
    Uninstalling dacite-1.9.2:
      Successfully uninstalled dacite-1.9.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [3]:
import sys
sys.path.append('/kaggle/usr/lib/notebooks/nikolozdodashvili/preprocessing/notebooks/nikolozdodashvili')

from preprocessing import (
    optimize_memory,
    DropHighMissingFeatures,
    FrequencyEncoder,
    MissingValueImputer,
    TimeFeatureExtractor,
    TransactionAmtTransformer,
    GroupAggregator,
    DropCorrelatedFeatures
)

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
train = train_transaction.merge(train_identity, on='TransactionID', how='left')

X = train.drop(columns=['isFraud', 'TransactionID'])
y = train['isFraud'].astype('int8')

X = optimize_memory(X)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

მეხსიერება დამუშავებამდე: 1946.36 MB
მეხსიერება დამუშავების შემდეგ: 921.47 MB
შემცირდა: 52.7%


In [5]:
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.frequency_maps = {}

    def fit(self, X, y=None):
        cat_cols = X.select_dtypes(include=['object', 'category']).columns
        for col in cat_cols:
            self.frequency_maps[col] = X[col].value_counts(normalize=True).to_dict()
        return self

    def transform(self, X):
        X = X.copy()
        for col, freq_map in self.frequency_maps.items():
            if col in X.columns:
                if hasattr(X[col], 'cat'):
                    X[col] = X[col].astype('object')
                X[col] = X[col].map(freq_map).fillna(0)
        return X

# TRAIN AND MLFLOW TRACKING

In [6]:
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

dagshub.init(repo_owner='ndoda23', 
             repo_name='MachineLearning---IEEE-CIS-Fraud-Detection', 
             mlflow=True)

mlflow.set_experiment("DecisionTree_Training")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=2df24495-e82e-4d2c-bc17-4b9c06d4d2e1&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=b125c092a4da91edae4a088dc072afdd42579da9ac05dbbc8a00949f3711d76d




Accessing as ndoda23

Initialized MLflow to track repo "ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection"

Repository ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection initialized!

<Experiment: artifact_location='mlflow-artifacts:/1bb026104c2a4d0e8fab864eb5a3f7fd', creation_time=1778066408440, experiment_id='1', last_update_time=1778066408440, lifecycle_stage='active', name='DecisionTree_Training', tags={}, trace_location=None, workspace='default'>

In [7]:
# ===== Cleaning Run =====
with mlflow.start_run(run_name="DecisionTree_Cleaning"):
    cleaner = DropHighMissingFeatures(threshold=0.9)
    cleaner.fit(X_train)
    mlflow.log_param("missing_threshold", 0.9)
    mlflow.log_metric("dropped_columns", len(cleaner.features_to_drop_))
    mlflow.log_metric("remaining_columns", X_train.shape[1] - len(cleaner.features_to_drop_))

Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
🏃 View run DecisionTree_Cleaning at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/f0127b575a844f1381dd5b201de3c2d9
🧪 View experiment at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1


In [8]:
# ===== Feature Engineering Run =====
with mlflow.start_run(run_name="DecisionTree_Feature_Engineering"):
    mlflow.log_param("encoding", "FrequencyEncoding")
    mlflow.log_param("imputer_strategy", "median")
    mlflow.log_param("time_features", True)
    mlflow.log_param("amt_features", True)
    mlflow.log_param("group_aggregations", "card1, addr1")
    mlflow.log_metric("features_added", 13)

🏃 View run DecisionTree_Feature_Engineering at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/d469a4bbfa2147e088aa54b8d130a568
🧪 View experiment at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1


In [9]:
# Run 1 - max_depth=5 
with mlflow.start_run(run_name="DecisionTree_Training_run1"):
    pipeline = Pipeline([
        ('cleaner',  DropHighMissingFeatures(threshold=0.9)),
        ('encoder',  FrequencyEncoder()),
        ('imputer',  MissingValueImputer(strategy='median')),
        ('time',     TimeFeatureExtractor()),
        ('amt',      TransactionAmtTransformer()),
        ('group',    GroupAggregator()),
        ('selector', DropCorrelatedFeatures(threshold=0.92)),
        ('model',    DecisionTreeClassifier(max_depth=5, random_state=42))
    ])
    pipeline.fit(X_train, y_train)

    y_pred_proba = pipeline.predict_proba(X_val)[:, 1]
    y_pred_class = pipeline.predict(X_val)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='roc_auc')

    mlflow.log_param("max_depth", 5)
    mlflow.log_param("imputer_strategy", "median")
    mlflow.log_param("corr_threshold", 0.92)
    mlflow.log_metric("val_roc_auc",  roc_auc_score(y_val, y_pred_proba))
    mlflow.log_metric("val_pr_auc",   average_precision_score(y_val, y_pred_proba))
    mlflow.log_metric("val_f1",       f1_score(y_val, y_pred_class))
    mlflow.log_metric("val_precision",precision_score(y_val, y_pred_class))
    mlflow.log_metric("val_recall",   recall_score(y_val, y_pred_class))
    mlflow.log_metric("cv_auc_mean",  cv_scores.mean())
    mlflow.log_metric("cv_auc_std",   cv_scores.std())
    mlflow.sklearn.log_model(pipeline, "pipeline")

    print(f"Run 1 (max_depth=5 - underfitting):")
    print(f"ROC AUC: {roc_auc_score(y_val, y_pred_proba):.4f}")
    print(f"PR AUC:  {average_precision_score(y_val, y_pred_proba):.4f}")
    print(f"F1:      {f1_score(y_val, y_pred_class):.4f}")
    print(f"CV AUC:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 165
დარჩება 268 სვეტი
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 163
დარჩება 270 სვეტი
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 170
დარჩება 263 სვეტი
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 167
დარჩება 266 სვეტი


2026/05/06 14:32:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 14:33:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 1 (max_depth=5 - underfitting):
ROC AUC: 0.7400
PR AUC:  0.2995
F1:      0.3758
CV AUC:  0.7439 ± 0.0028
🏃 View run DecisionTree_Training_run1 at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/1134ead80e164c95be7e9652dc789739
🧪 View experiment at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1


In [18]:
existing_run_id = "1134ead80e164c95be7e9652dc789739"  

with mlflow.start_run(run_id=existing_run_id):
    y_train_proba = pipeline.predict_proba(X_train)[:, 1]
    y_train_class = pipeline.predict(X_train)
    
    train_auc = roc_auc_score(y_train, y_train_proba)
    val_auc   = roc_auc_score(y_val, y_pred_proba)
    
    mlflow.log_metric("train_roc_auc",   train_auc)
    mlflow.log_metric("train_f1",        f1_score(y_train, y_train_class))
    mlflow.log_metric("train_precision", precision_score(y_train, y_train_class))
    mlflow.log_metric("train_recall",    recall_score(y_train, y_train_class))
    mlflow.log_metric("overfit_gap",     train_auc - val_auc)

🏃 View run DecisionTree_Training_run1 at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/1134ead80e164c95be7e9652dc789739
🧪 View experiment at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1


In [ ]:
with mlflow.start_run(run_id=existing_run_id):
    cv_train_scores = cross_val_score(pipeline, X_train, y_train, 
                                       cv=3, scoring='roc_auc')
    
    mlflow.log_metric("train_roc_auc", cv_train_scores.mean())
    mlflow.log_metric("overfit_gap", cv_train_scores.mean() - 0.7399)
    
    print(f"Train ROC AUC (CV): {cv_train_scores.mean():.4f}")

In [11]:
# Run 2 - max_depth=None (overfitting საჩვენებლად)
with mlflow.start_run(run_name="DecisionTree_Training_run2"):
    pipeline = Pipeline([
        ('cleaner',  DropHighMissingFeatures(threshold=0.9)),
        ('encoder',  FrequencyEncoder()),
        ('imputer',  MissingValueImputer(strategy='median')),
        ('time',     TimeFeatureExtractor()),
        ('amt',      TransactionAmtTransformer()),
        ('group',    GroupAggregator()),
        ('selector', DropCorrelatedFeatures(threshold=0.92)),
        ('model',    DecisionTreeClassifier(max_depth=None, random_state=42))
    ])
    pipeline.fit(X_train, y_train)

    y_pred_proba = pipeline.predict_proba(X_val)[:, 1]
    y_pred_class = pipeline.predict(X_val)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='roc_auc')

    mlflow.log_param("max_depth", "None")
    mlflow.log_param("imputer_strategy", "median")
    mlflow.log_param("corr_threshold", 0.92)
    mlflow.log_metric("val_roc_auc",  roc_auc_score(y_val, y_pred_proba))
    mlflow.log_metric("val_pr_auc",   average_precision_score(y_val, y_pred_proba))
    mlflow.log_metric("val_f1",       f1_score(y_val, y_pred_class))
    mlflow.log_metric("val_precision",precision_score(y_val, y_pred_class))
    mlflow.log_metric("val_recall",   recall_score(y_val, y_pred_class))
    mlflow.log_metric("cv_auc_mean",  cv_scores.mean())
    mlflow.log_metric("cv_auc_std",   cv_scores.std())
    mlflow.sklearn.log_model(pipeline, "pipeline")

    print(f"Run 2 (max_depth=None - overfitting):")
    print(f"ROC AUC: {roc_auc_score(y_val, y_pred_proba):.4f}")
    print(f"PR AUC:  {average_precision_score(y_val, y_pred_proba):.4f}")
    print(f"F1:      {f1_score(y_val, y_pred_class):.4f}")
    print(f"CV AUC:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
🏃 View run DecisionTree_Training_run2 at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/9e8e060775e54bacbc2f86030f6fbfc0
🧪 View experiment at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1


KeyboardInterrupt: 

In [ ]:
y_train_proba = pipeline.predict_proba(X_train)[:, 1]
y_train_class = pipeline.predict(X_train)

In [ ]:
with mlflow.start_run(run_id="e9ae5cb7ab90490cbee41e7b20b34983"):
    mlflow.log_metric("train_roc_auc",  roc_auc_score(y_train, y_train_proba))
    mlflow.log_metric("train_val_diff", roc_auc_score(y_train, y_train_proba) - roc_auc_score(y_val, y_pred_proba))
    mlflow.log_metric("train_f1",       f1_score(y_train, y_train_class))
    mlflow.log_metric("train_recall",   recall_score(y_train, y_train_class))

In [19]:
# Run 3 - max_depth=15, min_samples_split=50 (საუკეთესო ბალანსი)
with mlflow.start_run(run_name="DecisionTree_Training_run3"):
    pipeline = Pipeline([
        ('cleaner',  DropHighMissingFeatures(threshold=0.9)),
        ('encoder',  FrequencyEncoder()),
        ('imputer',  MissingValueImputer(strategy='median')),
        ('time',     TimeFeatureExtractor()),
        ('amt',      TransactionAmtTransformer()),
        ('group',    GroupAggregator()),
        ('selector', DropCorrelatedFeatures(threshold=0.92)),
        ('model',    DecisionTreeClassifier(max_depth=15, 
                                            min_samples_split=50,
                                            min_samples_leaf=20,
                                            random_state=42))
    ])
    pipeline.fit(X_train, y_train)

    y_pred_proba = pipeline.predict_proba(X_val)[:, 1]
    y_pred_class = pipeline.predict(X_val)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='roc_auc')

    mlflow.log_param("max_depth", 15)
    mlflow.log_param("min_samples_split", 50)
    mlflow.log_param("min_samples_leaf", 20)
    mlflow.log_param("imputer_strategy", "median")
    mlflow.log_param("corr_threshold", 0.92)
    mlflow.log_metric("val_roc_auc",  roc_auc_score(y_val, y_pred_proba))
    mlflow.log_metric("val_pr_auc",   average_precision_score(y_val, y_pred_proba))
    mlflow.log_metric("val_f1",       f1_score(y_val, y_pred_class))
    mlflow.log_metric("val_precision",precision_score(y_val, y_pred_class))
    mlflow.log_metric("val_recall",   recall_score(y_val, y_pred_class))
    mlflow.log_metric("cv_auc_mean",  cv_scores.mean())
    mlflow.log_metric("cv_auc_std",   cv_scores.std())
    mlflow.sklearn.log_model(pipeline, "pipeline")

    print(f"Run 3 (max_depth=15, regularized):")
    print(f"ROC AUC: {roc_auc_score(y_val, y_pred_proba):.4f}")
    print(f"PR AUC:  {average_precision_score(y_val, y_pred_proba):.4f}")
    print(f"F1:      {f1_score(y_val, y_pred_class):.4f}")
    print(f"CV AUC:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 165
დარჩება 268 სვეტი
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 163
დარჩება 270 სვეტი
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 170
დარჩება 263 სვეტი
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 167
დარჩება 266 სვეტი


2026/05/06 15:13:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:13:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 3 (max_depth=15, regularized):
ROC AUC: 0.8572
PR AUC:  0.5040
F1:      0.5039
CV AUC:  0.8485 ± 0.0036
🏃 View run DecisionTree_Training_run3 at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/bf2111429c434baba4db0511bde2782f
🧪 View experiment at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1


In [20]:
existing_run_id_run3 = "bf2111429c434baba4db0511bde2782f"

with mlflow.start_run(run_id=existing_run_id_run3):
    y_train_proba = pipeline.predict_proba(X_train)[:, 1]
    y_train_class = pipeline.predict(X_train)
    
    train_auc = roc_auc_score(y_train, y_train_proba)
    val_auc   = roc_auc_score(y_val, y_pred_proba)
    
    mlflow.log_metric("train_roc_auc",   train_auc)
    mlflow.log_metric("train_f1",        f1_score(y_train, y_train_class))
    mlflow.log_metric("train_precision", precision_score(y_train, y_train_class))
    mlflow.log_metric("train_recall",    recall_score(y_train, y_train_class))
    mlflow.log_metric("overfit_gap",     train_auc - val_auc)

🏃 View run DecisionTree_Training_run3 at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/bf2111429c434baba4db0511bde2782f
🧪 View experiment at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1


In [ ]:
with mlflow.start_run(run_id="10d27e8d7d544d708d65d225efbbc747"):
    mlflow.log_metric("train_roc_auc",  roc_auc_score(y_train, y_train_proba))
    mlflow.log_metric("train_val_diff", roc_auc_score(y_train, y_train_proba) - roc_auc_score(y_val, y_pred_proba))
    mlflow.log_metric("train_f1",       f1_score(y_train, y_train_class))
    mlflow.log_metric("train_recall",   recall_score(y_train, y_train_class))